In [8]:
import pandas as pd
import openpyxl

pora_data_dirty = pd.read_csv('Data/Data_Sberindex_POAD_1.csv', encoding='windows-1251', sep=';')
population = pd.read_parquet('Data/Rosstat/rosstat/rosstat/2_bdmo_population.parquet')
migration = pd.read_parquet('Data/Rosstat/rosstat/rosstat/3_bdmo_migration.parquet')
salary = pd.read_parquet('Data/Rosstat/rosstat/rosstat/4_bdmo_salary.parquet')
consumption = pd.read_parquet('Data/hackathonlicence/consumption.parquet')
market_access = pd.read_parquet('Data/hackathonlicence/market_access.parquet')
territory_id_pora = pd.read_csv('Data/territory_id_pora.csv', encoding='windows-1251', sep=';')

pora_data = pora_data_dirty[pora_data_dirty['arctic'] == True]

pora_data = pd.merge(pora_data, territory_id_pora, on=['region','municipality_up_name','municipality_down_name','settlement_name','settlement_name_sep','type'], how='left')

In [25]:
pora_data.head()

,region,municipality_up_name,municipality_down_name,settlement_name,settlement_name_sep,type,arctic,remote,special,suburb,...,2.1.2_healthcare_new,2.1.3_housing_new,2.1.4_sports_new,2.1.7_public_spaces_new,2.1.8_education_new,2.2.1_air_quality_new,2.3.5_suitability_indigenous_binary_new,2.7_crime_experience_binary_new,municipality_up_name_actual,territory_id
0,Ямало-Ненецкий автономный округ,Муниципальный округ Пуровский район,Муниципальный округ Пуровский район,пгт Уренгой,пгт Уренгой (Ямало-Ненецкий автономный округ),пгт,True,False,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Муниципальный округ Пуровский район,2882
1,Ненецкий автономный округ,Заполярный муниципальный район,Городское поселение рабочий поселок Искателей,рабочий поселок Искателей,рабочий поселок Искателей (Ненецкий автономный...,рп,True,False,0,True,...,"3,8","3,7","3,6","3,7","3,8","3,3","0,9","0,2",Заполярный муниципальный район,917
2,Ямало-Ненецкий автономный округ,Городской округ город Лабытнанги,Городской округ город Лабытнанги,пгт Харп,пгт Харп (Ямало-Ненецкий автономный округ),пгт,True,False,0,True,...,"2,1","2,5","3,0","2,9","3,4","3,8","0,8","0,4",Городской округ город Лабытнанги,2880
3,Архангельская область,Пинежский муниципальный район,Карпогорское сельское поселение,село Карпогоры,село Карпогоры (Архангельская область),село,True,False,ОНП,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Пинежский муниципальный округ,910
4,Мурманская область,Городской округ город Североморск (ЗАТО),Городской округ город Североморск (ЗАТО),пгт Сафоново,пгт Сафоново (Мурманская область),пгт,True,False,0,True,...,"3,1","2,5","2,6","2,4","3,4","3,5","0,8","0,3",городской округ ЗАТО город Североморск,1517


In [12]:
# Агрегация данных по населению: получим общую численность по территории и году
population_agg = population.groupby(['territory_id','year'])['value'].sum().reset_index()
population_agg.rename(columns={'value':'total_population'}, inplace=True)

# Агрегация данных по миграции: получим сальдо миграции по территории и году
migration_agg = migration.groupby(['territory_id','year'])['value'].sum().reset_index()
migration_agg.rename(columns={'value':'migration_balance'}, inplace=True)

# Объединим население и миграцию
migration_pop = pd.merge(migration_agg, population_agg, on=['territory_id','year'], how='left')
migration_pop['migration_rate'] = migration_pop['migration_balance'] / migration_pop['total_population'] * 1000

full_data = pd.merge(migration_pop, pora_data, on=['territory_id'], how='inner')

# full_data.to_csv(
#     'Data/arctic_municipalities_data.csv', 
#     index=False, 
#     encoding='utf-8-sig',
#     sep=';',  # разделитель точка с запятой (лучше для Excel)
#     decimal=',',  # десятичный разделитель запятая
#     quotechar='"',  # символ кавычек
#     quoting=1  # заключать в кавычки все нечисловые поля
# )

full_data.to_excel('arctic_municipalities_data.xlsx', index=False, sheet_name='Арктические_муниципалитеты')

full_data.shape



(46, 100)